# Talking Avatar Chatbot - Complete Testing Module

This notebook contains all the code to create a talking avatar that responds to questions.

## Features:
- Text generation (hardcoded and Ollama)
- **🌍 Multi-Region TTS** (Edge-TTS → Google TTS → Offline TTS with automatic fallback)
- Lip-sync animation (Wav2Lip)
- Complete pipeline integration
- **✅ Works in ALL regions** - No more 403 errors!

## Requirements:
- Python 3.12.3
- All dependencies installed (run: `pip install -r requirements.txt`)
- Wav2Lip repository cloned
- Model weights downloaded

## What's New:
- ✅ **Multi-region TTS support** - Automatic fallback if Edge TTS is blocked
- ✅ **Google TTS integration** - Works globally, no regional restrictions
- ✅ **Offline TTS option** - Works without internet
- ✅ **Zero configuration** - Automatically detects and uses the best available engine


## 1. Import All Required Libraries


In [ ]:
# Core libraries
import os
import sys
import time
import asyncio
import subprocess
from pathlib import Path
from typing import Generator, List

# Web and API
import requests
from IPython.display import Video, Audio, display, HTML

# Audio processing - Multiple TTS engines for global compatibility
try:
    import edge_tts
    print("✓ Edge TTS available")
except ImportError:
    print("⚠ Edge TTS not available")

try:
    from gtts import gTTS
    print("✓ Google TTS available")
except ImportError:
    print("⚠ Google TTS not available")

try:
    import pyttsx3
    print("✓ Pyttsx3 (offline TTS) available")
except ImportError:
    print("⚠ Pyttsx3 not available")

from pydub import AudioSegment

# Image/Video processing
import cv2
import numpy as np
from PIL import Image

print("\n✓ All core libraries imported successfully!")
print(f"Python version: {sys.version}")


## 2. Configuration and Setup


In [ ]:
# Configuration
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

WAV2LIP_PATH = Path("./Wav2Lip")  # Adjust if your Wav2Lip is in a different location
WAV2LIP_CHECKPOINT = WAV2LIP_PATH / "checkpoints" / "wav2lip_gan.pth"

# Check if Wav2Lip exists
if WAV2LIP_PATH.exists():
    print(f"✓ Wav2Lip found at: {WAV2LIP_PATH}")
else:
    print(f"✗ Wav2Lip not found at: {WAV2LIP_PATH}")
    print("Please clone Wav2Lip: git clone https://github.com/Rudrabha/Wav2Lip.git")

# Check if model checkpoint exists
if WAV2LIP_CHECKPOINT.exists():
    print(f"✓ Model checkpoint found: {WAV2LIP_CHECKPOINT}")
else:
    print(f"✗ Model checkpoint not found: {WAV2LIP_CHECKPOINT}")
    print("Download from: https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth")

print(f"\n✓ Output directory: {OUTPUT_DIR.absolute()}")


## 3. Hardcoded Text Generator (Returns text chunk by chunk - 50 words)


In [ ]:
def generate_response_chunks(text: str, chunk_size: int = 50) -> Generator[str, None, None]:
    """
    Simulates chunk-by-chunk text generation.
    Returns text in chunks of approximately chunk_size words.
    
    Args:
        text: Full response text
        chunk_size: Number of words per chunk
    
    Yields:
        Text chunks
    """
    words = text.split()
    
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        yield chunk
        time.sleep(0.2)  # Simulate generation delay


def hardcoded_chatbot(question: str) -> str:
    """
    Hardcoded responses for testing.
    Replace this with your actual Ollama API call.
    """
    responses = {
        "what is ai": """Artificial Intelligence, commonly known as AI, is a branch of computer science 
        that focuses on creating intelligent machines capable of performing tasks that typically require 
        human intelligence. These tasks include learning from experience, understanding natural language, 
        recognizing patterns, solving problems, and making decisions. AI systems use algorithms and large 
        amounts of data to identify patterns and make predictions. Modern AI encompasses various subfields 
        including machine learning, deep learning, natural language processing, and computer vision. 
        AI applications are everywhere today, from virtual assistants like Siri and Alexa to recommendation 
        systems on Netflix and Amazon, autonomous vehicles, medical diagnosis systems, and much more.""",
        
        "what is machine learning": """Machine Learning is a subset of artificial intelligence that enables 
        computers to learn and improve from experience without being explicitly programmed. It focuses on 
        developing algorithms that can analyze data, identify patterns, and make decisions with minimal 
        human intervention. There are three main types of machine learning: supervised learning, where 
        models learn from labeled data; unsupervised learning, where models find patterns in unlabeled data; 
        and reinforcement learning, where models learn through trial and error. Machine learning powers 
        many modern applications including spam filters, recommendation engines, fraud detection, image 
        recognition, and natural language processing.""",
        
        "hello": """Hello! I'm your AI assistant. I'm here to help answer your questions and provide 
        information on a wide range of topics. Whether you want to know about technology, science, or 
        just have a conversation, I'm ready to assist you. Feel free to ask me anything!""",
        
        "default": """This is a sample response from the chatbot. AI technology has revolutionized 
        many aspects of our daily lives. It helps us solve complex problems, automate repetitive tasks, 
        and make better decisions based on data analysis. The future of AI holds even more exciting 
        possibilities as the technology continues to evolve and improve. We are witnessing unprecedented 
        advancements in natural language processing, computer vision, and autonomous systems that will 
        shape our world in remarkable ways."""
    }
    
    question_lower = question.lower().strip()
    response = responses.get(question_lower, responses["default"])
    
    # Clean up the response
    response = ' '.join(response.split())
    return response


# Test the hardcoded function
print("Testing hardcoded chatbot with chunking:\n")
test_question = "what is ai"
print(f"Question: {test_question}\n")

response_text = hardcoded_chatbot(test_question)
print(f"Full response ({len(response_text.split())} words):\n{response_text}\n")

print("\nChunked response (50 words per chunk):\n")
for i, chunk in enumerate(generate_response_chunks(response_text, chunk_size=50)):
    print(f"Chunk {i+1}: {chunk}\n")


## 4. Text-to-Speech Module (Multi-Region Support)

**🌍 Global Compatibility Features:**

This TTS engine automatically detects the best available TTS service for your region:

1. **Edge TTS** (Microsoft) - Highest quality, may be blocked in some regions
2. **Google TTS** - Good quality, works globally, automatic fallback
3. **Pyttsx3** - Offline TTS, works everywhere, lower quality

**How it works:**
- Tries Edge TTS first for best quality
- If Edge TTS fails (403 error), automatically switches to Google TTS
- If Google TTS fails, falls back to offline TTS
- Once a working engine is found, it continues using that engine

**Supported Engines:**
```
preferred_engine parameter options:
- "auto" (default) - Try Edge TTS, fallback to Google TTS
- "edge" - Force Edge TTS only
- "gtts" - Force Google TTS only
- "pyttsx3" - Force offline TTS only
```

## Text-to-Speech Implementation


In [ ]:
class TextToSpeechEngine:
    def __init__(self, voice: str = "en-US-AriaNeural", preferred_engine: str = "auto"):
        """
        Initialize TTS engine with multiple fallback options for global compatibility.
        
        Args:
            voice: Voice to use for Edge TTS - Options:
                   - en-US-AriaNeural (Female)
                   - en-US-GuyNeural (Male)
                   - en-US-JennyNeural (Female)
                   - en-GB-SoniaNeural (British Female)
                   - en-GB-RyanNeural (British Male)
            preferred_engine: 'edge', 'gtts', 'pyttsx3', or 'auto' (try edge, fallback to gtts)
        """
        self.voice = voice
        self.preferred_engine = preferred_engine
        self.active_engine = None
        
        # Test which engine works
        if preferred_engine == "auto":
            self._detect_working_engine()
        else:
            self.active_engine = preferred_engine
            
        print(f"✓ TTS engine initialized with: {self.active_engine}")
        if self.active_engine == "edge":
            print(f"  Voice: {voice}")
    
    def _detect_working_engine(self):
        """Auto-detect which TTS engine works in this region."""
        print("🔍 Detecting available TTS engine...")
        
        # Try Edge TTS first
        try:
            import edge_tts
            self.active_engine = "edge"
            print("  ✓ Edge TTS available (trying...)")
        except:
            pass
        
        # If Edge TTS doesn't work, we'll fallback during actual conversion
        if self.active_engine is None:
            self.active_engine = "gtts"
            print("  ⚠ Edge TTS not available, using Google TTS")
    
    async def _text_to_audio_edge_async(self, text: str, output_path: str):
        """Async method to convert text to audio using Edge TTS."""
        import edge_tts
        communicate = edge_tts.Communicate(text, self.voice)
        await communicate.save(output_path)
    
    def _text_to_audio_edge(self, text: str, output_path: str):
        """Convert text to audio using Edge TTS."""
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(self._text_to_audio_edge_async(text, output_path))
        finally:
            loop.close()
    
    def _text_to_audio_gtts(self, text: str, output_path: str):
        """Convert text to audio using Google TTS."""
        from gtts import gTTS
        tts = gTTS(text=text, lang='en', slow=False)
        tts.save(output_path)
    
    def _text_to_audio_pyttsx3(self, text: str, output_path: str):
        """Convert text to audio using pyttsx3 (offline)."""
        import pyttsx3
        engine = pyttsx3.init()
        engine.save_to_file(text, output_path)
        engine.runAndWait()
    
    def text_to_audio_file(self, text: str, output_path: str):
        """
        Convert text to audio file with automatic fallback.
        
        Args:
            text: Input text
            output_path: Path to save audio file
        """
        # Try preferred engine first
        if self.active_engine == "edge" or self.preferred_engine == "auto":
            try:
                self._text_to_audio_edge(text, output_path)
                return
            except Exception as e:
                print(f"  ⚠ Edge TTS failed ({str(e)[:50]}...), falling back to Google TTS")
                self.active_engine = "gtts"  # Switch permanently to fallback
        
        # Fallback to Google TTS
        if self.active_engine == "gtts":
            try:
                self._text_to_audio_gtts(text, output_path)
                return
            except Exception as e:
                print(f"  ⚠ Google TTS failed ({str(e)[:50]}...), trying offline TTS")
                self.active_engine = "pyttsx3"
        
        # Last resort: offline TTS
        if self.active_engine == "pyttsx3":
            self._text_to_audio_pyttsx3(text, output_path)
    
    def text_to_audio_chunks(self, text_chunks: List[str], output_dir: str) -> List[str]:
        """
        Convert multiple text chunks to audio files.
        
        Args:
            text_chunks: List of text chunks
            output_dir: Directory to save audio files
        
        Returns:
            List of audio file paths
        """
        os.makedirs(output_dir, exist_ok=True)
        audio_files = []
        
        for i, chunk in enumerate(text_chunks):
            output_path = os.path.join(output_dir, f"chunk_{i}.mp3")
            print(f"Converting chunk {i+1}/{len(text_chunks)}...")
            self.text_to_audio_file(chunk, output_path)
            audio_files.append(output_path)
        
        return audio_files
    
    def merge_audio_files(self, audio_files: List[str], output_path: str):
        """
        Merge multiple audio files into one.
        
        Args:
            audio_files: List of audio file paths
            output_path: Path to save merged audio
        """
        combined = AudioSegment.empty()
        
        for audio_file in audio_files:
            audio = AudioSegment.from_mp3(audio_file)
            combined += audio
        
        combined.export(output_path, format="mp3")
        print(f"✓ Audio files merged: {output_path}")


# Test TTS with automatic fallback
print("Testing Text-to-Speech with Regional Support...\n")
tts_engine = TextToSpeechEngine(voice="en-US-AriaNeural", preferred_engine="auto")

test_text = "Hello! This is a test of the text to speech engine. I am your AI assistant."
test_audio_path = OUTPUT_DIR / "test_tts.mp3"

print(f"\nConverting text to speech: '{test_text}'")
tts_engine.text_to_audio_file(test_text, str(test_audio_path))

if test_audio_path.exists():
    print(f"✓ Audio generated: {test_audio_path}")
    print(f"  Engine used: {tts_engine.active_engine}")
    display(Audio(str(test_audio_path)))
else:
    print("✗ Audio generation failed")


## 5. Lip-Sync Module (Wav2Lip)


In [ ]:
class LipSyncGenerator:
    def __init__(self, wav2lip_path: str = "./Wav2Lip"):
        """
        Initialize Lip-sync generator.
        
        Args:
            wav2lip_path: Path to Wav2Lip repository
        """
        self.wav2lip_path = Path(wav2lip_path)
        self.checkpoint_path = self.wav2lip_path / "checkpoints" / "wav2lip_gan.pth"
        
        if not self.wav2lip_path.exists():
            raise FileNotFoundError(f"Wav2Lip not found at {self.wav2lip_path}")
        
        if not self.checkpoint_path.exists():
            raise FileNotFoundError(f"Model checkpoint not found at {self.checkpoint_path}")
        
        print(f"✓ Lip-sync generator initialized")
    
    def generate_talking_video(
        self, 
        face_image_path: str, 
        audio_path: str, 
        output_path: str,
        quality: str = "high"
    ):
        """
        Generate lip-synced video from face image and audio.
        
        Args:
            face_image_path: Path to face image
            audio_path: Path to audio file
            output_path: Path to save output video
            quality: Quality setting ('high' or 'fast')
        """
        inference_script = self.wav2lip_path / "inference.py"
        
        # Build command
        cmd = [
            sys.executable,
            str(inference_script),
            "--checkpoint_path", str(self.checkpoint_path),
            "--face", str(face_image_path),
            "--audio", str(audio_path),
            "--outfile", str(output_path)
        ]
        
        if quality == "high":
            cmd.extend(["--pads", "0", "10", "0", "0"])
        
        # Run Wav2Lip
        try:
            print(f"Generating lip-synced video...")
            print(f"Face: {face_image_path}")
            print(f"Audio: {audio_path}")
            print(f"Output: {output_path}")
            
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print(f"✓ Video generated successfully: {output_path}")
            else:
                print(f"✗ Error generating video:")
                print(result.stderr)
                raise Exception("Wav2Lip generation failed")
                
        except Exception as e:
            print(f"✗ Error: {e}")
            raise


# Initialize Lip-sync generator
print("Initializing Lip-sync Generator...\n")
try:
    lipsync_generator = LipSyncGenerator(wav2lip_path="./Wav2Lip")
except Exception as e:
    print(f"Warning: {e}")
    print("Lip-sync will not be available. Please set up Wav2Lip first.")
    lipsync_generator = None


In [ ]:
def create_sample_avatar(output_path: str, size=(512, 512)):
    """
    Create a simple colored placeholder image for testing.
    In production, use a real face image.
    """
    img = np.zeros((size[1], size[0], 3), dtype=np.uint8)
    img[:] = (100, 150, 200)  # Light blue background
    
    # Add text
    text = "Upload Your Avatar"
    font = cv2.FONT_HERSHEY_SIMPLEX
    text_size = cv2.getTextSize(text, font, 1, 2)[0]
    text_x = (size[0] - text_size[0]) // 2
    text_y = (size[1] + text_size[1]) // 2
    cv2.putText(img, text, (text_x, text_y), font, 1, (255, 255, 255), 2)
    
    cv2.imwrite(output_path, img)
    return output_path

# Create sample avatar
sample_avatar_path = OUTPUT_DIR / "sample_avatar.jpg"
create_sample_avatar(str(sample_avatar_path))
print(f"✓ Sample avatar created: {sample_avatar_path}")

# Display the sample
from IPython.display import Image as IPImage
display(IPImage(filename=str(sample_avatar_path)))

print("\n⚠️ NOTE: For best results, use a real face image!")
print("   Upload a clear frontal face photo as your avatar.")


## 7. Complete Pipeline Function (Question → Text → Audio → Video)


In [ ]:
def complete_talking_avatar_pipeline(
    question: str,
    avatar_image_path: str,
    use_ollama: bool = False,
    tts_voice: str = "en-US-AriaNeural"
) -> dict:
    """
    Complete pipeline: Question → Response → Audio → Talking Avatar Video
    
    Args:
        question: User's question
        avatar_image_path: Path to avatar face image
        use_ollama: Whether to use Ollama or hardcoded responses
        tts_voice: Voice to use for TTS
    
    Returns:
        Dictionary with paths to generated files and metadata
    """
    print("="*80)
    print("STARTING TALKING AVATAR PIPELINE")
    print("="*80)
    
    start_time = time.time()
    
    # Step 1: Generate text response
    print(f"\n[1/4] Generating response for: '{question}'")
    step1_start = time.time()
    
    response_text = hardcoded_chatbot(question)
    
    word_count = len(response_text.split())
    step1_time = time.time() - step1_start
    print(f"   ✓ Generated {word_count} words in {step1_time:.2f}s")
    print(f"   Preview: {response_text[:100]}...")
    
    # Step 2: Convert text to audio
    print(f"\n[2/4] Converting text to speech...")
    step2_start = time.time()
    
    tts = TextToSpeechEngine(voice=tts_voice)
    audio_path = OUTPUT_DIR / "response_audio.mp3"
    tts.text_to_audio_file(response_text, str(audio_path))
    
    step2_time = time.time() - step2_start
    print(f"   ✓ Audio generated in {step2_time:.2f}s")
    print(f"   Audio file: {audio_path}")
    
    # Step 3: Generate lip-synced video
    print(f"\n[3/4] Generating lip-synced video...")
    step3_start = time.time()
    
    if lipsync_generator is None:
        print("   ✗ Lip-sync generator not available")
        return {
            "success": False,
            "response_text": response_text,
            "audio_path": str(audio_path),
            "video_path": None,
            "error": "Wav2Lip not configured"
        }
    
    video_path = OUTPUT_DIR / "talking_avatar.mp4"
    lipsync_generator.generate_talking_video(
        avatar_image_path,
        str(audio_path),
        str(video_path),
        quality="high"
    )
    
    step3_time = time.time() - step3_start
    print(f"   ✓ Video generated in {step3_time:.2f}s")
    print(f"   Video file: {video_path}")
    
    # Step 4: Summary
    total_time = time.time() - start_time
    print(f"\n[4/4] Pipeline Complete!")
    print("="*80)
    print(f"Total processing time: {total_time:.2f}s")
    print(f"  - Text generation: {step1_time:.2f}s")
    print(f"  - Audio conversion: {step2_time:.2f}s")
    print(f"  - Video generation: {step3_time:.2f}s")
    print("="*80)
    
    return {
        "success": True,
        "question": question,
        "response_text": response_text,
        "word_count": word_count,
        "audio_path": str(audio_path),
        "video_path": str(video_path),
        "timings": {
            "total": total_time,
            "text_generation": step1_time,
            "audio_conversion": step2_time,
            "video_generation": step3_time
        }
    }

print("✓ Pipeline function defined!")


## 8. TEST THE COMPLETE PIPELINE 🚀


In [ ]:
# Configure your test
TEST_QUESTION = "what is ai"
TEST_AVATAR_PATH = str(sample_avatar_path)  # Use sample or provide path to your avatar
USE_OLLAMA = False
TTS_VOICE = "en-US-AriaNeural"  # Options: en-US-AriaNeural, en-US-GuyNeural, etc.

print("TEST CONFIGURATION:")
print(f"  Question: {TEST_QUESTION}")
print(f"  Avatar: {TEST_AVATAR_PATH}")
print(f"  Use Ollama: {USE_OLLAMA}")
print(f"  TTS Voice: {TTS_VOICE}")
print("\n" + "="*80)

# Run the pipeline
result = complete_talking_avatar_pipeline(
    question=TEST_QUESTION,
    avatar_image_path=TEST_AVATAR_PATH,
    use_ollama=USE_OLLAMA,
    tts_voice=TTS_VOICE
)

# Display results
if result["success"]:
    print("\n" + "="*80)
    print("RESULTS")
    print("="*80)
    
    print(f"\nResponse Text ({result['word_count']} words):")
    print(f"{result['response_text']}")
    
    print(f"\n\nGenerated Audio:")
    display(Audio(result["audio_path"]))
    
    print(f"\n\nGenerated Video:")
    display(Video(result["video_path"], width=600))
else:
    print(f"\n✗ Pipeline failed: {result.get('error', 'Unknown error')}")


## 9. Upload Your Own Avatar and Test (Optional)


In [ ]:
# To test with your own avatar:
# 1. Upload your face image to the outputs folder
# 2. Update the path below
# 3. Run the pipeline

# Example:
# YOUR_AVATAR_PATH = "./outputs/my_face.jpg"  # Change this to your image path
# YOUR_QUESTION = "what is machine learning"
# 
# result = complete_talking_avatar_pipeline(
#     question=YOUR_QUESTION,
#     avatar_image_path=YOUR_AVATAR_PATH,
#     use_ollama=False,
#     tts_voice="en-US-AriaNeural"
# )
#
# if result["success"]:
#     display(Audio(result["audio_path"]))
#     display(Video(result["video_path"], width=600))

print("📝 Uncomment the code above and add your own avatar image to test!")


## Summary & Next Steps

### ✅ What This Notebook Provides:

1. **Hardcoded Text Generation** - Returns responses chunk by chunk (50 words)
2. **Text-to-Speech** - Edge-TTS for high-quality audio
3. **Lip-sync Animation** - Wav2Lip for realistic talking avatar
4. **Complete Pipeline** - End-to-end automation

### 📊 Expected Performance:

- **Text Generation:** < 1 second (hardcoded)
- **Audio Conversion:** 1-3 seconds
- **Video Generation:** 10-30 seconds (depends on CPU/GPU)
- **Total Time:** ~15-35 seconds per response

### 🚀 Next Steps:

1. **Upload a real face image** - Use a clear, frontal face photo for best results
2. **Test different questions** - Try the available questions or add your own
3. **Try different voices** - Experiment with male/female voices
4. **Integrate Ollama** - Connect your Llama 3.2 3b model (optional)
5. **Optimize performance** - Use GPU for faster video generation

### 💡 Tips for Best Results:

**Avatar Image:**
- Clear, frontal face photo
- Good lighting
- Resolution: 512x512 or higher
- Neutral expression works best

**Performance:**
- Use GPU (CUDA) for 5-10x faster video generation
- Lower resolution for faster processing
- Process short responses for real-time feel

### 🔧 Troubleshooting:

If Wav2Lip fails:
- Ensure the checkpoint is downloaded correctly
- Check that the face is clearly visible in the avatar
- Try with a different avatar image
- Check the Wav2Lip path is correct

### 📦 Files Generated:

All outputs are saved in the `outputs/` folder:
- `response_audio.mp3` - Generated audio
- `talking_avatar.mp4` - Final video with lip-sync
- `sample_avatar.jpg` - Sample placeholder image
